# Exercise 2: Predicting Repeat Purchases — Olist E-Commerce

## Table of Contents

- [0. Context](#toc-context)
- [1. Building the feature table (SQL)](#toc-1)
- [2. Load and split](#toc-2)
- [3. Integrity check before fitting](#toc-3)
- [4. Baseline model](#toc-4)
- [5. Error Analysis](#toc-5)
- [6. Model Comparison](#toc-6)
- [7. Interpretability](#toc-7)
- [8. Conclusion](#toc-8)

*Draft shape, same as Exercise 1's at this stage: expect it to evolve once the error analysis shows what actually needs its own section.*

<a id="toc-context"></a>

## 0. Context

You've joined the growth team at Olist, a Brazilian marketplace that connects small merchants to major retail channels. Marketing wants to know which first-time customers are likely to place a second order, so retention spend goes to the customers actually worth targeting, not blanket discounts to everyone.

### Data handed over

Olist's own public, anonymized order data (2016-2018), released for research and practice: about 100k orders spread across 9 related tables, not one clean feature table.

- **orders**: one row per order, with status and key timestamps (purchase, approval, delivery)
- **order_items**: line items per order (product, seller, price, freight)
- **order_payments**: payment method and value, possibly several per order
- **order_reviews**: post-purchase review score and text
- **customers**: one row per customer record, with location
- **products**: product category and physical dimensions
- **sellers**: seller location
- **geolocation**: zip-code-level latitude/longitude
- **product_category_name_translation**: category names, Portuguese to English

Unlike Exercise 1, no feature table exists yet: building one correctly from these tables is itself part of the job.

**Target.** Whether a first-time customer goes on to place a second order within some follow-up window after their first. Most customers in this dataset never return, so this already looks like an imbalanced classification problem; confirming that, and deciding the exact follow-up window and cutoff, are among the first things to settle once the data's loaded.

### Objective

Build a model that scores a first-time customer's likelihood of returning, using only information knowable at the time of their first order, nothing from after it. Report how good the model actually is, and be explicit about what "good" means for an imbalanced target before trusting any single number.

### What's different from Exercise 1

Three concrete differences, worth naming up front rather than discovering midway:

**1. There's no feature table yet.** In Exercise 1, `fetch_california_housing()` handed over one table: one row per block group, one column per input, ready to fit. A "feature table" just means exactly that: one row per thing we're predicting about, one column per input the model sees. Here, the raw data is 9 tables at different granularities (one row per order, one row per item, one row per payment, ...), none of them shaped one-row-per-customer. Building that table, by joining these raw tables and aggregating them (counting orders, summing payments, computing time gaps) down to one row per customer, is work that has to happen before any model touches the data. Exercise 1 never needed this step; its one table already had the right shape.

**2. The target is a category, not a number.** Exercise 1 predicted a price, any real number; a residual told us "off by how much." Here the target is **did this customer return, or not**, exactly two categories. Linear regression doesn't apply: nothing keeps its output between 0 and 1, or makes it mean anything as a probability. **Logistic regression** is built for exactly this: instead of predicting the value directly, it predicts the *probability* that a row belongs to the "returned" category. The "fit a line, read its coefficients" intuition carries over, but the cost function underneath changes (log loss, not squared error) to match a probability output instead of a continuous one.

**3. SQL builds the table.** The raw tables are relational: an order links to a customer via `customer_id`, to its items via `order_id`, and so on. Reconstructing "how many orders has this customer placed, how much did they spend, how long before their second order" from rows scattered across several tables is exactly what SQL's `JOIN` (combine tables via a shared key) and `GROUP BY` (collapse many rows into one summary row per group, e.g. per customer) are built to do. It's also how this would actually happen at a company: a data warehouse holds the raw tables, SQL turns them into a model-ready table, and only then does Python take over for modeling. We'll use **DuckDB**, a SQL engine that runs embedded in the notebook, no server to set up, and can query a CSV file or a pandas DataFrame directly by name.

### Approach

Past those two differences, the same process as Exercise 1: a simple baseline first (logistic regression), diagnose its errors before deciding anything, let the evidence, likely the class imbalance, drive the metric choice, then compare models, probably ending with gradient-boosted trees given how well ensembles performed last time, plus SHAP for interpretability, a gap worth closing for this kind of work.

One new rule alongside that process: **point-in-time correctness**. Every feature can only use information available strictly before the prediction moment (a customer's first order); the label comes from strictly after it. Getting this wrong is a subtler, time-based version of the train/test leakage Exercise 1 was careful about.

### A first look at SQL, before loading the real data

Two tiny, hand-built stand-ins for two of Olist's real tables (`orders` and
`order_payments`), small enough to read by eye, just to see how a SQL
query is written and called before pointing the same kind of query at the
real, much bigger tables.

In [1]:
import duckdb
import pandas as pd

# same column names and shape as the real tables, just 4 made-up rows instead of ~100k
orders = pd.DataFrame({
    "order_id":                 ["o1", "o2", "o3", "o4"],
    "customer_id":              ["c1", "c1", "c2", "c3"],
    "order_purchase_timestamp": pd.to_datetime(["2017-01-05", "2017-03-20", "2017-02-10", "2017-04-01"]),
})

order_payments = pd.DataFrame({
    "order_id":      ["o1", "o2", "o3", "o4"],
    "payment_value": [120.50, 45.00, 300.00, 80.00],
})

print(f"orders:\n{orders}")
print(f"\norder_payments:\n{order_payments}")

orders:
  order_id customer_id order_purchase_timestamp
0       o1          c1               2017-01-05
1       o2          c1               2017-03-20
2       o3          c2               2017-02-10
3       o4          c3               2017-04-01

order_payments:
  order_id  payment_value
0       o1          120.5
1       o2           45.0
2       o3          300.0
3       o4           80.0


In [2]:
# DuckDB sees orders/order_payments directly by variable name, no export step needed
query = """
SELECT
    o.customer_id,
    COUNT(*)             AS n_orders,
    SUM(p.payment_value) AS total_spent
FROM orders AS o
JOIN order_payments AS p ON o.order_id = p.order_id
GROUP BY o.customer_id
ORDER BY o.customer_id
"""
customer_summary = duckdb.sql(query).df()
print(customer_summary)

  customer_id  n_orders  total_spent
0          c1         2        165.5
1          c2         1        300.0
2          c3         1         80.0


Reading the query clause by clause: `JOIN ... ON o.order_id = p.order_id`
lines up each order with its payment using the key they share, `GROUP BY
o.customer_id` then collapses the (possibly several) matched rows down to
one row per customer, and `COUNT(*)`/`SUM(...)` compute a number per group
rather than per row. The result: `c1` (2 orders) has `n_orders = 2` and
`total_spent = 165.50`, `c2` and `c3` (1 order each) keep their single
order's values. That is, in miniature, exactly the operation needed to
turn Olist's real tables into one row per customer.

### Loading the real tables

The 9 CSVs come from Kaggle via `kagglehub`, which caches them locally (no account needed for a public dataset), so the data never lives in the repo. Each CSV is then loaded into DuckDB as a table, under a short name (`olist_orders_dataset.csv` becomes `orders`), so the real queries can be written exactly like the toy one above.

In [3]:
import os
import kagglehub

# download once (later runs reuse the local cache), pinned to version 2 so the data can't change under the analysis
data_dir = kagglehub.dataset_download("olistbr/brazilian-ecommerce/versions/2")

# load each CSV as a DuckDB table, short name = file name minus the "olist_" prefix and "_dataset" suffix
for file in sorted(os.listdir(data_dir)):
    name = file.removesuffix(".csv").removeprefix("olist_").removesuffix("_dataset")
    duckdb.sql(f"CREATE OR REPLACE TABLE {name} AS SELECT * FROM read_csv_auto('{os.path.join(data_dir, file)}')")

# confirm all 9 tables loaded, with their row counts
print(duckdb.sql("SELECT table_name, estimated_size AS n_rows FROM duckdb_tables() ORDER BY table_name"))


┌───────────────────────────────────┬─────────┐
│            table_name             │ n_rows  │
│              varchar              │  int64  │
├───────────────────────────────────┼─────────┤
│ customers                         │   99441 │
│ geolocation                       │ 1000163 │
│ order_items                       │  112650 │
│ order_payments                    │  103886 │
│ order_reviews                     │   99224 │
│ orders                            │   99441 │
│ product_category_name_translation │      71 │
│ products                          │   32951 │
│ sellers                           │    3095 │
└───────────────────────────────────┴─────────┘

